In [26]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [27]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)
training_set = train_datagen.flow_from_directory(
    '../data/chest_xray/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training',
    shuffle=True
)

Found 4173 images belonging to 2 classes.


Test set prep

In [28]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_set = test_datagen.flow_from_directory(
        '../data/chest_xray/test',
        target_size=(224,224),
        batch_size=32,
        class_mode='binary',
        shuffle=False)

Found 624 images belonging to 2 classes.


Validation set prep

In [29]:
validation_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

validation_set = validation_datagen.flow_from_directory(
    '../data/chest_xray/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

Found 1043 images belonging to 2 classes.


In [30]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(training_set.classes),
    y=training_set.classes
)

class_weights = dict(enumerate(class_weights))

print("Class indices:", training_set.class_indices)
print("Class weights:", class_weights)

Class indices: {'NORMAL': 0, 'PNEUMONIA': 1}
Class weights: {0: np.float64(1.9445479962721341), 1: np.float64(0.6730645161290323)}


BUILDING CNN

In [31]:
cnn=tf.keras.models.Sequential()

In [32]:
cnn.add(tf.keras.layers.Conv2D(
    filters=32,
    kernel_size=3,
    activation='relu',
    input_shape=[224, 224, 3]
))

c:\Users\Srijan Yadav\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [33]:
cnn.add(tf.keras.layers.BatchNormalization())

In [34]:
cnn.add(tf.keras.layers.MaxPool2D(
    pool_size=2,
    strides=2
))

In [36]:
cnn.add(tf.keras.layers.Conv2D(
    filters=64,
    kernel_size=3,
    activation='relu'
))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(
    pool_size=2,
    strides=2
))

In [37]:
cnn.add(tf.keras.layers.Conv2D(
    filters=128,
    kernel_size=3,
    activation='relu'
))
cnn.add(tf.keras.layers.BatchNormalization())

cnn.add(tf.keras.layers.MaxPool2D(
    pool_size=2,
    strides=2
))

In [38]:
cnn.add(tf.keras.layers.GlobalAveragePooling2D())

Dense layer

In [39]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

In [40]:

cnn.add(tf.keras.layers.Dropout(0.5))

cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [41]:
cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [42]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [43]:
cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 52, 52, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 147,969 (578.00 KB)

 Trainable params: 147,393 (575.75 KB)

 Non-trainable params: 576 (2.25 KB)

BALANCING

In [44]:
history = cnn.fit(
    x=training_set,
    validation_data=validation_set,
    epochs=25,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/25
131/131 ━━━━━━━━━━━━━━━━━━━━ 99s 745ms/step - accuracy: 0.8397 - loss: 0.3554 - val_accuracy: 0.7430 - val_loss: 2.7874
Epoch 2/25
131/131 ━━━━━━━━━━━━━━━━━━━━ 94s 719ms/step - accuracy: 0.8756 - loss: 0.2965 - val_accuracy: 0.7430 - val_loss: 1.9181
Epoch 3/25
131/131 ━━━━━━━━━━━━━━━━━━━━ 94s 717ms/step - accuracy: 0.8843 - loss: 0.2647 - val_accuracy: 0.7430 - val_loss: 5.2188
Epoch 4/25
131/131 ━━━━━━━━━━━━━━━━━━━━ 95s 721ms/step - accuracy: 0.8840 - loss: 0.2733 - val_accuracy: 0.7430 - val_loss: 1.9467
Epoch 5/25
131/131 ━━━━━━━━━━━━━━━━━━━━ 97s 738ms/step - accuracy: 0.9075 - loss: 0.2271 - val_accuracy: 0.6635 - val_loss: 0.6954
Epoch 6/25
131/131 ━━━━━━━━━━━━━━━━━━━━ 95s 722ms/step - accuracy: 0.9061 - loss: 0.2274 - val_accuracy: 0.7430 - val_loss: 3.3910
Epoch 7/25
131/131 ━━━━━━━━━━━━━━━━━━━━ 92s 700ms/step - accuracy: 0.9087 - loss: 0.2183 - val_accuracy: 0.7430 - val_loss: 1.0692
Epoch 8/25
131/131 ━━━━━━━━━━━━━━━━━━━━ 91s 695ms/step - accuracy: 0.9202 - loss: 0

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Training','Validation'])
plt.show()



In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Model Loss")
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Training', 'Validation'])
plt.show()


In [ ]:
test_loss, test_accuracy = cnn.evaluate(test_set)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
import numpy as np

test_set.reset()

predictions = cnn.predict(test_set)

predicted_classes = (predictions > 0.5).astype(int).flatten()

true_classes = test_set.classes

In [ ]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(true_classes, predicted_classes)
print(cm)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(
    true_classes,
    predicted_classes,
    target_names=['NORMAL', 'PNEUMONIA']
))

In [ ]:
print(predictions[:20].flatten())

print("Minimum:", predictions.min())
print("Maximum:", predictions.max())
print("Mean:", predictions.mean())